<a href="https://colab.research.google.com/github/dewmini-06/Statistical-Learning-e22317/blob/main/Assignment_7b_E22317.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 4 – Gaussian Mixture Clustering as Conditional Updating

---

## 1. Deriving the Marginal Density

Using the law of total probability,

$$
p(x_i)=\sum_{k=1}^{K}P(C_i=k)\,P(x_i|C_i=k).
$$

Since

$$
P(C_i=k)=\phi_k
$$

and

$$
X_i|C_i=k\sim N(\mu_k,\Sigma_k),
$$

the marginal density becomes

$$
\boxed{
p(x_i)=\sum_{k=1}^{K}\phi_k
N(x_i|\mu_k,\Sigma_k)
}
$$

### Explanation

This is called a **Gaussian Mixture Density** because the overall probability distribution is formed by combining several Gaussian distributions, each weighted by its corresponding mixture probability $\phi_k$. Each Gaussian represents one cluster, and together they model complex, multimodal datasets.

---

## 2. Posterior Cluster Probability

Applying Bayes' theorem,

$$
P(C_i=k|X_i=x_i)
=
\frac{P(X_i=x_i|C_i=k)\,P(C_i=k)}
{\sum_{j=1}^{K}P(X_i=x_i|C_i=j)\,P(C_i=j)}.
$$

Substituting the Gaussian model,

$$
\boxed{
\gamma_{ik}
=
\frac{\phi_kN(x_i|\mu_k,\Sigma_k)}
{\sum_{j=1}^{K}\phi_jN(x_i|\mu_j,\Sigma_j)}
}
$$

where

$$
\gamma_{ik}
=
P(C_i=k|X_i=x_i).
$$

### Explanation

The responsibility $\gamma_{ik}$ is the posterior probability that observation $x_i$ belongs to cluster $k$. A larger value indicates stronger evidence that the observation belongs to that cluster.

---

## 3. One-Hot Encoding of the Latent Variable

Define the latent vector

$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$

where

$$
Z_{ik}
=
\begin{cases}
1, & C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$

Then

$$
E[Z_{ik}|X_i=x_i]
=
P(C_i=k|X_i=x_i)
=
\gamma_{ik}.
$$

Hence,

$$
\boxed{
E[Z_i|X_i=x_i]
=
\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}
}
$$

### Explanation

The conditional expectation of the one-hot encoded latent variable equals the vector of posterior probabilities. Therefore, the soft cluster assignment is exactly the conditional expectation of the latent membership variable.

---

## 4. Soft vs Hard Clustering

Hard clustering assigns each observation to exactly one cluster:

$$
\boxed{
\hat{C}_i
=
\arg\max_{1\le k\le K}\gamma_{ik}
}
$$

whereas soft clustering assigns a probability to every cluster.

### Difference

- **Soft clustering:** Each observation has a probability of belonging to every cluster.
- **Hard clustering:** Each observation is assigned only to the cluster with the highest posterior probability.

Soft clustering preserves uncertainty, whereas hard clustering makes a single definitive assignment.

---

## 5. Conditional Expectation of the Observation

For a Gaussian distribution,

$$
\boxed{
E[X_i|C_i=k]
=
\mu_k
}
$$

The vector $\mu_k$ is therefore the center (mean) of cluster $k$.

Comparison:

$$
E[Z_i|X_i=x_i]
=
\gamma_i
$$

describes the probability of cluster membership for an observation,

whereas

$$
E[X_i|C_i=k]
=
\mu_k
$$

gives the expected location of observations within cluster $k$.

Thus, the first represents **cluster membership probabilities**, while the second represents the **cluster center**.

---

## 6. Complete-Data Likelihood

The complete-data likelihood is

$$
p(x,z)
=
\prod_{i=1}^{n}
\prod_{k=1}^{K}
\left[
\phi_k
N(x_i|\mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$

Taking the logarithm,

$$
\boxed{
\ell_c
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
z_{ik}
\left[
\log\phi_k
+
\log N(x_i|\mu_k,\Sigma_k)
\right]
}
$$

### Explanation

If the cluster labels were known, maximizing the complete-data log-likelihood would be straightforward because each observation would already belong to a specific Gaussian component.

---

## 7. EM Interpretation

Since the cluster labels are unknown,

$$
z_{ik}
\rightarrow
E[Z_{ik}|X_i=x_i]
=
\gamma_{ik}.
$$

Therefore,

$$
\boxed{
Q
=
\sum_{i=1}^{n}
\sum_{k=1}^{K}
\gamma_{ik}
\left[
\log\phi_k
+
\log N(x_i|\mu_k,\Sigma_k)
\right]
}
$$

### Explanation

The **Expectation (E)-step** computes the posterior probabilities (responsibilities) for each observation. These probabilities replace the unknown cluster labels and represent the current belief about cluster membership.

---

## 8. Parameter Updates

The effective number of observations assigned to cluster $k$ is

$$
\boxed{
N_k
=
\sum_{i=1}^{n}\gamma_{ik}
}
$$

The updated parameters are

### Mixture Weight

$$
\boxed{
\phi_k^{\text{new}}
=
\frac{N_k}{n}
}
$$

### Mean

$$
\boxed{
\mu_k^{\text{new}}
=
\frac{\sum_{i=1}^{n}\gamma_{ik}x_i}
{N_k}
}
$$

### Covariance

$$
\boxed{
\Sigma_k^{\text{new}}
=
\frac{
\sum_{i=1}^{n}
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T
}
{N_k}
}
$$

### Explanation

The responsibility $\gamma_{ik}$ acts as a fractional membership weight. Instead of assigning each observation completely to one cluster, every data point contributes proportionally to each cluster according to its posterior probability.

---

## 9. Interpretation

Gaussian Mixture Model (GMM) clustering can be viewed as a repeated process of conditional updating. Initially, each cluster has a prior probability represented by its mixture weight. After observing a data point, Bayes' theorem is used to compute the posterior probability (responsibility) that the point belongs to each cluster. These posterior probabilities provide soft cluster assignments, allowing each observation to partially belong to multiple clusters. During the M-step, the mixture weights, cluster means, and covariance matrices are updated using these posterior probabilities as fractional weights. Repeating the E-step and M-step enables the model to refine both the cluster memberships and the model parameters until convergence, producing an accurate probabilistic clustering of the data.

---

## 10. Computational Simulation and Out-of-Sample Validation

### Analysis

The Gaussian Mixture Model successfully separates the dataset into three probabilistic clusters using the Expectation-Maximization (EM) algorithm. The density heatmap reveals the underlying distribution of the training data, while the contour plots illustrate the decision boundaries between clusters. Training and test observations located near the cluster centers have high posterior responsibilities and are assigned with high confidence. Points close to the decision boundaries exhibit mixed responsibilities, indicating uncertainty in cluster membership. A high average log-likelihood on the test dataset demonstrates that the learned Gaussian distributions generalize well to unseen data. Overall, the interactive visualizations confirm that GMM performs soft probabilistic clustering by continuously updating posterior cluster membership probabilities throughout the EM algorithm.